# Sanskrit → English NMT — **Evaluation / Inference Notebook**

**This is the file to run at evaluation time.** It does **not** train anything. It reloads the model
saved by `train.ipynb` and produces the final `submission.csv` (and metrics, if references are
available) on whatever test file you point it at — including the private set released in class.

### What it needs (from `train.ipynb`, committed to the repo)
`artifacts/config.json`, `artifacts/best_model.pt`, `artifacts/spm_sa.model`, `artifacts/spm_en.model`.

### What you set
Just the paths in the **Configuration** cell: the artifacts folder and the test source CSV
(and, if you have them, the reference CSV for scoring).

**Disclosure:** the translation model uses no pre-trained weights. BERTScore internally loads a
pre-trained RoBERTa-large model to compute its metric only.

## 0. Install dependencies

In [ ]:
%pip install -q torch sentencepiece nltk bert-score pandas

## 1. Configuration — **edit these paths**

At evaluation time, set `TEST_SA_PATH` to the released private Sanskrit file. If a reference English
file is provided, set `TEST_EN_PATH` too and metrics will be computed; otherwise leave it `None` and
the notebook just writes `submission.csv`.

In [ ]:
import os, math, json, time, unicodedata
from types import SimpleNamespace

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# --- paths ---
ARTIFACTS_DIR = "artifacts"                 # folder committed from train.ipynb
TEST_SA_PATH  = "test_sa_1000.csv"          # <-- private Sanskrit test file at eval time
TEST_EN_PATH  = "test_en_1000.csv"          # <-- reference file, or None if not provided
OUTPUT_CSV    = "submission.csv"

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = pick_device()
print("Device:", DEVICE)

## 2. Load the saved config and tokenizers

`cfg` is rebuilt from `config.json` as a simple namespace with exactly the fields the model and
decoder read. You can override `beam_size` / `length_penalty` here if you want a speed/quality
trade-off different from training.

In [ ]:
import sentencepiece as spm

with open(os.path.join(ARTIFACTS_DIR, "config.json")) as f:
    cfg_dict = json.load(f)
cfg = SimpleNamespace(**cfg_dict)

# Optional overrides for decoding at eval time:
# cfg.beam_size = 5
# cfg.length_penalty = 0.6
cfg.batch_size = getattr(cfg, "batch_size", 64)

sp_src = spm.SentencePieceProcessor(); sp_src.load(os.path.join(ARTIFACTS_DIR, "spm_sa.model"))
sp_tgt = spm.SentencePieceProcessor(); sp_tgt.load(os.path.join(ARTIFACTS_DIR, "spm_en.model"))
print(f"Loaded config + tokenizers. vocab SA={cfg.src_vocab}, EN={cfg.tgt_vocab}, "
      f"beam={cfg.beam_size}")

## 3. Rebuild the model and load the trained weights

The architecture code below is identical to `train.ipynb` — it must be, so the saved weights load
cleanly. We build the model from `cfg`, then load `best_model.pt`.

In [ ]:
class PositionalEncoding(nn.Module):
    """Adds a fixed sinusoidal signal so the model can tell positions apart."""
    def __init__(self, d_model, max_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.h = num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        split = lambda x: x.view(B, -1, self.h, self.d_k).transpose(1, 2)
        q, k, v = split(self.q_proj(query)), split(self.k_proj(key)), split(self.v_proj(value))

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        attn = self.dropout(torch.softmax(scores, dim=-1))
        ctx = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, -1, self.h * self.d_k)
        return self.out_proj(ctx)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model),
        )
    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, h, src_mask))
        h = self.norm2(x)
        x = x + self.dropout(self.ff(h))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask, src_mask):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, h, tgt_mask))
        h = self.norm2(x)
        x = x + self.dropout(self.cross_attn(h, memory, memory, src_mask))
        h = self.norm3(x)
        x = x + self.dropout(self.ff(h))
        return x


class TransformerNMT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        self.src_embed = nn.Embedding(cfg.src_vocab, d, padding_idx=cfg.pad_id)
        self.tgt_embed = nn.Embedding(cfg.tgt_vocab, d, padding_idx=cfg.pad_id)
        self.pos = PositionalEncoding(d, cfg.max_len, cfg.dropout)
        self.encoder = nn.ModuleList(
            [EncoderLayer(d, cfg.num_heads, cfg.d_ff, cfg.dropout) for _ in range(cfg.num_layers)])
        self.decoder = nn.ModuleList(
            [DecoderLayer(d, cfg.num_heads, cfg.d_ff, cfg.dropout) for _ in range(cfg.num_layers)])
        self.enc_norm = nn.LayerNorm(d)
        self.dec_norm = nn.LayerNorm(d)
        self.generator = nn.Linear(d, cfg.tgt_vocab)
        self.generator.weight = self.tgt_embed.weight   # weight tying
        self._init()

    def _init(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_src_mask(self, src):
        return (src == self.cfg.pad_id).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        T = tgt.size(1)
        pad = (tgt == self.cfg.pad_id).unsqueeze(1).unsqueeze(2)
        causal = torch.triu(torch.ones(T, T, device=tgt.device), 1).bool()
        return pad | causal.unsqueeze(0).unsqueeze(1)

    def encode(self, src, src_mask):
        x = self.pos(self.src_embed(src) * math.sqrt(self.cfg.d_model))
        for layer in self.encoder:
            x = layer(x, src_mask)
        return self.enc_norm(x)

    def decode(self, tgt, memory, tgt_mask, src_mask):
        x = self.pos(self.tgt_embed(tgt) * math.sqrt(self.cfg.d_model))
        for layer in self.decoder:
            x = layer(x, memory, tgt_mask, src_mask)
        return self.dec_norm(x)

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        memory = self.encode(src, src_mask)
        out = self.decode(tgt, memory, tgt_mask, src_mask)
        return self.generator(out)

model = TransformerNMT(cfg).to(DEVICE)
state = torch.load(os.path.join(ARTIFACTS_DIR, "best_model.pt"), map_location=DEVICE)
model.load_state_dict(state)
model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded model — {total_params:,} parameters")

## 4. Load and tokenize the test set

Light cleaning (identical to training), then subword encoding. References are optional: if
`TEST_EN_PATH` is set and the file exists, we load them for scoring.

In [ ]:
def find_csv(data_dir, split, side):
    """Locate a CSV for a given split ('train'/'dev'/'test') and side ('sa'/'en'),
    tolerant of suffixes like _10000 / _1000 in the provided filenames."""
    import glob
    hits = []
    for f in glob.glob(os.path.join(data_dir, "*.csv")):
        name = os.path.basename(f).lower()
        side_ok = (f"_{side}_" in name) or (f"_{side}." in name) or name.endswith(f"{side}.csv")
        if split in name and side_ok:
            hits.append(f)
    if not hits:
        raise FileNotFoundError(f"No CSV found for split='{split}', side='{side}' in {data_dir}")
    return sorted(hits)[0]


def _find_col(df, *keywords):
    for c in df.columns:
        name = c.strip().lower().replace(" ", "_")
        if all(k in name for k in keywords):
            return c
    raise KeyError(f"No column matching {keywords} in {list(df.columns)}")


def clean(text):
    text = unicodedata.normalize("NFC", str(text))
    text = " ".join(text.split())      # collapse runs of whitespace/newlines/tabs
    return text.strip()


def load_side(path, side):
    df = pd.read_csv(path)
    id_col = _find_col(df, "id")
    txt_col = _find_col(df, "sentence", side)
    return pd.DataFrame({"id": df[id_col], "text": df[txt_col].map(clean)})

src_df = load_side(TEST_SA_PATH, "sa").rename(columns={"text": "src"})
HAS_REF = TEST_EN_PATH is not None and os.path.exists(TEST_EN_PATH)
if HAS_REF:
    ref_df = load_side(TEST_EN_PATH, "en").rename(columns={"text": "tgt"})
    data_df = src_df.merge(ref_df, on="id", how="inner").reset_index(drop=True)
else:
    data_df = src_df.copy(); data_df["tgt"] = None
print(f"test sentences: {len(data_df)} | references available: {HAS_REF}")


class TranslationDataset(Dataset):
    def __init__(self, df, sp_src, max_len):
        self.rows = [(r["id"], sp_src.encode(r["src"], out_type=int)[: max_len - 2], None)
                     for _, r in df.iterrows()]
    def __len__(self):  return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def pad_batch(seqs, pad_id):
    m = max(len(s) for s in seqs)
    return torch.tensor([s + [pad_id] * (m - len(s)) for s in seqs], dtype=torch.long)

def collate_infer(batch):
    ids, src, _ = zip(*batch)
    return list(ids), pad_batch(list(src), cfg.pad_id)

test_ds = TranslationDataset(data_df, sp_src, cfg.max_len)

## 5. Decoding

Same greedy + beam-search implementation as training. `translate_corpus` runs the whole set and
returns `{id: english}`.

In [ ]:
@torch.no_grad()
def greedy_decode_batch(model, src, max_len):
    model.eval()
    src = src.to(DEVICE)
    src_mask = model.make_src_mask(src)
    memory = model.encode(src, src_mask)
    B = src.size(0)
    ys = torch.full((B, 1), cfg.bos_id, dtype=torch.long, device=DEVICE)
    done = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    for _ in range(max_len - 1):
        tgt_mask = model.make_tgt_mask(ys)
        out = model.decode(ys, memory, tgt_mask, src_mask)
        nxt = model.generator(out[:, -1]).argmax(-1, keepdim=True)
        ys = torch.cat([ys, nxt], dim=1)
        done = done | (nxt.squeeze(1) == cfg.eos_id)
        if done.all():
            break
    return ys


@torch.no_grad()
def beam_decode_one(model, src_ids, max_len, beam_size, length_penalty):
    model.eval()
    src = src_ids.unsqueeze(0).to(DEVICE)
    src_mask = model.make_src_mask(src)
    memory = model.encode(src, src_mask)

    beams = [(torch.tensor([cfg.bos_id], device=DEVICE), 0.0)]
    finished = []
    for _ in range(max_len - 1):
        pool = []
        for seq, score in beams:
            if seq[-1].item() == cfg.eos_id:
                finished.append((seq, score)); continue
            tgt_mask = model.make_tgt_mask(seq.unsqueeze(0))
            out = model.decode(seq.unsqueeze(0), memory, tgt_mask, src_mask)
            logp = F.log_softmax(model.generator(out[:, -1]), dim=-1).squeeze(0)
            vals, idx = logp.topk(beam_size)
            for v, i in zip(vals, idx):
                pool.append((torch.cat([seq, i.view(1)]), score + v.item()))
        if not pool:
            break
        norm = lambda x: x[1] / (len(x[0]) ** length_penalty)
        pool.sort(key=norm, reverse=True)
        beams = pool[:beam_size]
        if all(s[-1].item() == cfg.eos_id for s, _ in beams):
            finished.extend(beams); break
    pool = finished if finished else beams
    pool.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
    return pool[0][0]


def ids_to_text(ids, sp):
    """Strip special tokens and detokenize back to a normal string."""
    out = []
    for t in (ids.tolist() if torch.is_tensor(ids) else ids):
        if t == cfg.eos_id:
            break
        if t not in (cfg.bos_id, cfg.pad_id):
            out.append(t)
    return sp.decode(out)


def translate_corpus(model, dataset, use_beam=True):
    """Translate an entire dataset, returning a dict {id: english_string}."""
    preds = {}
    loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_infer)
    for ids, src in loader:
        if use_beam:
            for i, one in zip(ids, src):
                real = one[one != cfg.pad_id]
                out = beam_decode_one(model, real, cfg.max_len, cfg.beam_size, cfg.length_penalty)
                preds[i] = ids_to_text(out, sp_tgt)
        else:
            out = greedy_decode_batch(model, src, cfg.max_len)
            for i, row in zip(ids, out):
                preds[i] = ids_to_text(row, sp_tgt)
    return preds

## 6. Translate the test set, timed, and write `submission.csv`

The wall-clock time here is the efficiency figure to report. Beam search is the default; for a
faster run pass `use_beam=False` (greedy) — quality usually drops a little.

In [ ]:
start = time.time()
preds = translate_corpus(model, test_ds, use_beam=True)
inference_time = time.time() - start
print(f"Translated {len(preds)} sentences in {inference_time:.2f}s "
      f"({inference_time/len(preds)*1000:.1f} ms/sentence)")

order = {sid: k for k, sid in enumerate(data_df["id"].tolist())}
submission = (pd.DataFrame({"Source_id": list(preds.keys()),
                            "Sentence_en": list(preds.values())})
              .sort_values("Source_id", key=lambda s: s.map(order))
              .reset_index(drop=True))
submission.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print("Wrote", OUTPUT_CSV, "with", len(submission), "rows")
submission.head()

## 7. Metrics (only if references are available)

BLEU (NLTK, default weights) and F1 BERTScore (`rescale_with_baseline=True`, `lang="en"`), plus the
efficiency figures. On the private set this is where your final scores come from.

In [ ]:
import nltk
from nltk.translate.bleu_score import corpus_bleu

def compute_corpus_bleu(pred_dict, ref_df):
    """Default-weight NLTK corpus BLEU between predictions and references, aligned by id."""
    refs, hyps = [], []
    for _, r in ref_df.iterrows():
        if r["id"] in pred_dict:
            refs.append([str(r["tgt"]).split()])   # list-of-references form
            hyps.append(pred_dict[r["id"]].split())
    if not hyps:
        return 0.0
    return corpus_bleu(refs, hyps)   # default weights (0.25 x4) as specified

print(f"Total parameters : {total_params:,}")
print(f"Inference time   : {inference_time:.2f} s  (beam={cfg.beam_size})")

if HAS_REF:
    from bert_score import score as bertscore_score
    bleu = compute_corpus_bleu(preds, data_df)
    ids  = [i for i in data_df["id"] if i in preds]
    cands = [preds[i] for i in ids]
    refs  = [str(data_df.loc[data_df["id"] == i, "tgt"].iloc[0]) for i in ids]
    _, _, F1 = bertscore_score(cands, refs, lang="en", rescale_with_baseline=True)
    print(f"BLEU             : {bleu*100:.2f}")
    print(f"BERTScore F1     : {F1.mean().item():.4f}")
else:
    print("No references provided — submission.csv written; metrics skipped.")

---
**Reproducibility.** This notebook is deterministic given the committed artifacts: same weights,
same tokenizers, same decoding settings. Point `TEST_SA_PATH` at the released file and run top to
bottom.